## t1-1

In [ ]:
import numpy as np
from collections import defaultdict
# 定义常量
WEATHER_LIST = [
    '高温', '高温', '晴朗', '沙暴', '晴朗', '高温', '沙暴', '晴朗', '高温', '高温',
    '沙暴', '高温', '晴朗', '高温', '高温', '高温', '沙暴', '沙暴', '高温', '高温',
    '晴朗', '晴朗', '高温', '晴朗', '沙暴', '高温', '晴朗', '晴朗', '高温', '高温'
]

MINE_NODES = [12]
VILLAGE_NODES = [15]
DESTINATION_NODES = [27]

END_NODE = 27
MAX_DAY = 30
INITIAL_MONEY = 10000
WEIGHT_LIMIT = 1200
BASE_INCOME = 1000

# 资源参数
WATER_WEIGHT = 3
FOOD_WEIGHT = 2
WATER_PRICE = 5
FOOD_PRICE = 10
WATER_REFUND = 2.5
FOOD_REFUND = 5

# 基础消耗量 (晴朗, 高温, 沙暴)
BASE_CONSUMPTION = {
    '晴朗': (5, 7),
    '高温': (8, 6),
    '沙暴': (10, 10)
}

# 地图邻接表
GRAPH = {
    1: [25],
    25: [26],
    26: [23],
    23: [21],
    21: [27, 9],
    9: [15, 21],
    15: [13, 9],
    13: [12, 15],
    12: [13]
}

# 运动惯性路径定义
MOTION_INERTIA_PATHS = {
    # 路径1: 23-21-9-15-13-12
    (23, 21): 9,
    (21, 9): 15,
    (9, 15): 13,
    (15, 13): 12,
    # 路径2: 12-13-15 (反向)
    (12, 13): 15,
    (13, 15): 9,  # 15可以选择去9或13
    # 路径3: 15-9-21-27
    (15, 9): 21,
    (9, 21): 27
}

# 各节点到终点的最短天数
SHORTEST_DAYS_TO_END = {
    1: 3,
    9: 2,
    12: 5,
    13: 4,
    15: 3,
    21: 1,
    23: 2,
    25: 2,
    26: 1,
    27: 0
}

In [ ]:
def calculate_next_3move_need(day):
    """
    计算从指定日期开始，接下来三次移动所需的总物资（水和食物）。

    Args:
        day (int): 开始计算的日期 (1-indexed)。

    Returns:
        tuple: (总共需要的水, 总共需要的食物)。
               如果无法完成3次移动（例如，日期超出范围），则返回None。
    """
    total_water_needed = 0
    total_food_needed = 0
    moves_count = 0
    current_day_index = day + 1   # 调整为day之后一天

    while moves_count < 3:
        if current_day_index >= len(WEATHER_LIST):
            # 如果日期超出天气列表范围，则无法完成3次移动
            return None

        weather = WEATHER_LIST[current_day_index]
        base_water, base_food = BASE_CONSUMPTION[weather]

        if weather == '沙暴':
            # 沙暴天气，停留，消耗基础物资
            total_water_needed += base_water
            total_food_needed += base_food
        else:
            # 非沙暴天气，移动，消耗双倍物资
            total_water_needed += base_water * 2
            total_food_needed += base_food * 2
            moves_count += 1
        
        current_day_index += 1
        
    return total_water_needed, total_food_needed

In [14]:
def get_base_consumption(weather):
    return BASE_CONSUMPTION[weather]

def initialize_dp():
    dp = [dict() for _ in range(MAX_DAY + 1)]
    # 第0天：在起点购买资源
    for food in range(110, 350):
        for water in range(122, 435):
            if food < water: continue
            weight = WATER_WEIGHT * water + FOOD_WEIGHT * food
            
            # 无法再多带一箱水或食物
            if not (weight <= WEIGHT_LIMIT and weight + min(WATER_WEIGHT, FOOD_WEIGHT) > WEIGHT_LIMIT):
                continue
            
            cost = WATER_PRICE * water + FOOD_PRICE * food
            if cost <= INITIAL_MONEY:
                money = INITIAL_MONEY - cost
                state = (1, water, food)
                dp[0][state] = (money, None, None, None, None)
    return dp

def update_state(dp_next, state, money, pre_pos, pre_money, pre_water, pre_food):
    pos, water, food = state
    # 检查资源非负
    if water < 0 or food < 0:
        return
    # 检查负重限制
    weight = WATER_WEIGHT * water + FOOD_WEIGHT * food
    if weight > WEIGHT_LIMIT:
        return
    # 更新状态：保留资金最大的
    if state in dp_next:
        if money > dp_next[state][0]:
            dp_next[state] = (money, pre_pos, pre_money, pre_water, pre_food)
    else:
        dp_next[state] = (money, pre_pos, pre_money, pre_water, pre_food)

In [15]:
def _calculate_future_max_consumption():
    """
    Pre-calculates the maximum water and food needed from each day until the end.
    The consumption is estimated at 3 times the base rate for any weather.
    """
    future_max_consumption = {}
    # Start from the last day and go backwards
    max_needed_water = 0
    max_needed_food = 0
    for day in range(MAX_DAY - 1, -1, -1):
        # For day `d`, we need to calculate consumption for days `d+1` to `MAX_DAY-1`
        # The values for day `d` are the same as for `d+1` plus consumption on day `d+1`
        # But since we iterate backwards, we can just accumulate
        future_max_consumption[day] = (max_needed_water, max_needed_food)
        
        # Add consumption for the current day to be used for the previous day's calculation
        weather = WEATHER_LIST[day]
        bw, bf = get_base_consumption(weather)
        max_needed_water += 3 * bw
        max_needed_food += 3 * bf
        
    return future_max_consumption

# Pre-calculate the values once when the module is imported
FUTURE_MAX_CONSUMPTION = _calculate_future_max_consumption()
#FUTURE_MAX_CONSUMPTION


In [16]:
# dp = initialize_dp()
# dp

In [34]:
def simulate():
    dp = initialize_dp()
    global_max_money = -1
    
    for day in range(MAX_DAY):
        print(f"第{day+1}天")

        weather = WEATHER_LIST[day]
        base_water, base_food = get_base_consumption(weather)
        dp_next = dict()
        
        for state, (money, pre_pos, pre_money, pre_water, pre_food) in dp[day].items():
            pos, water, food = state
            if pos in DESTINATION_NODES: continue

            # 剪枝：如果剩余天数不足以到达终点
            if (day+1) + SHORTEST_DAYS_TO_END.get(pos, float('inf')) > MAX_DAY:
                continue
            
            # 沙暴日必须停留
            if weather == '沙暴':
                new_pos = pos
                consume_water = base_water
                consume_food = base_food
                new_water = water - consume_water
                new_food = food - consume_food
                new_money = money
                update_state(dp_next, (new_pos, new_water, new_food), new_money, pos, money, water, food)

                # 矿山挖矿选项
                if new_pos in MINE_NODES :# 挖矿
                    extra_water = 2 * base_water
                    extra_food = 2 * base_food
                    mine_water = new_water - extra_water
                    mine_food = new_food - extra_food
                    mine_money = new_money + BASE_INCOME
                    update_state(dp_next, (new_pos, mine_water, mine_food), mine_money, pos, money, water, food)
            
            # 非沙暴日：可选择停留或行走
            else:
                # 停留矿山挖矿选项
                if pos in MINE_NODES: # 挖矿
                    new_pos = pos
                    consume_water = base_water
                    consume_food = base_food
                    new_water = water - consume_water
                    new_food = food - consume_food
                    new_money = money

                    extra_water = 2 * base_water
                    extra_food = 2 * base_food
                    mine_water = new_water - extra_water
                    mine_food = new_food - extra_food
                    mine_money = new_money + BASE_INCOME
                    update_state(dp_next, (new_pos, mine_water, mine_food), mine_money, pos, money, water, food)

                # 选项2: 行走到相邻节点
                for neighbor in GRAPH[pos]:
                    # 运动惯性剪枝：如果有前一天位置，检查是否符合运动惯性
                    if pre_pos is not None:
                        # 检查是否在运动惯性路径上
                        if (pre_pos, pos) in MOTION_INERTIA_PATHS:
                            expected_next = MOTION_INERTIA_PATHS[(pre_pos, pos)]
                            # 如果有明确的下一个位置要求，且当前选择不符合，则跳过
                            if expected_next is not None and neighbor != expected_next:
                                continue
                    
                    new_pos = neighbor
                    consume_water = 2 * base_water
                    consume_food = 2 * base_food
                    new_water = water - consume_water
                    new_food = food - consume_food
                    new_money = money
                    
                    if new_pos == END_NODE and new_water >= 0 and new_food >= 0 and new_money >= 0:
                        final_money = new_money + WATER_REFUND * new_water + FOOD_REFUND * new_food
                        if final_money > global_max_money:
                            global_max_money = final_money 
                            print(f"新的最大收益: {global_max_money}")
                            print(f"{day+1}天")
                            print(f"新的最大收益状态: {new_pos, new_money, new_water, new_food}")
                            print(f"新的最大收益前一个状态: {pos, money, water, food}")
                        update_state(dp_next, (new_pos, new_water, new_food), new_money, pos, money, water, food)

                    else:
                        update_state(dp_next, (new_pos, new_water, new_food), new_money, pos, money, water, food)

                    # 村庄购买选项（行走到达当天不能挖矿）
                    if new_pos in VILLAGE_NODES and new_water >= 0 and new_food >= 0 and new_money >= 0: # 购买资源
                        if day == 7 : # 第8天特殊购买逻辑
                            # 目标：最大化负重，且满足水食比例
                            new_weight = WATER_WEIGHT * new_water + FOOD_WEIGHT * new_food
                            max_food = (WEIGHT_LIMIT - new_weight) // FOOD_WEIGHT
                            max_water = (WEIGHT_LIMIT - new_weight) // WATER_WEIGHT
                            for f in range(0, max_food + 1):
                                for w in range(0, max_water + 1):
                                    if f > w: continue
                                    total_water = new_water + w
                                    total_food = new_food + f
                                    
                                    weight = WATER_WEIGHT * total_water + FOOD_WEIGHT * total_food
                                    if not (weight <= WEIGHT_LIMIT and weight + min(WATER_WEIGHT, FOOD_WEIGHT) > WEIGHT_LIMIT): continue
                                        
                                    cost = 2 * WATER_PRICE * w + 2 * FOOD_PRICE * f
                                    if cost > new_money: continue
                                    
                                    buy_money = new_money - cost
                                    update_state(dp_next, (new_pos, total_water, total_food), buy_money, pos, money, water, food)

                        elif day >= 11:   # 非第8天 前往终点的途中 购买资源
                            need_w, need_f = calculate_next_3move_need(day)
                            buy_f = max(0, need_f - new_food)
                            buy_w = max(0, need_w - new_water)
                            total_water = new_water + buy_w
                            total_food = new_food + buy_f
                            weight = WATER_WEIGHT * total_water + FOOD_WEIGHT * total_food
                            if weight > WEIGHT_LIMIT: continue
                            cost = 2 * WATER_PRICE * buy_w + 2 * FOOD_PRICE * buy_f
                            if cost > new_money: continue
                            buy_money = new_money - cost
                            update_state(dp_next, (new_pos, total_water, total_food), buy_money, pos, money, water, food)


        dp[day + 1] = dp_next
        
        # if day + 1 == 8:
        #     for key, value in dp_next.items():
        #         print(key, value)
        #     break

    return global_max_money, dp

In [39]:
max_money, dp = simulate()
print(f"最大收益: {max_money}")
# print(f"dp: {dp}")



第1天
第2天
第3天
第4天
第5天
第6天
第7天
第8天
第9天
新的最大收益: 7365.0
9天
新的最大收益状态: (27, 6400, 126, 130)
新的最大收益前一个状态: (21, 6400, 142, 142)
第10天
第11天
第12天
第13天
第14天
第15天
第16天
新的最大收益: 7545.0
16天
新的最大收益状态: (27, 7400, 6, 26)
新的最大收益前一个状态: (21, 7400, 22, 38)
第17天
第18天
第19天
新的最大收益: 7860.0
19天
新的最大收益状态: (27, 7860, 0, 0)
新的最大收益前一个状态: (21, 7860, 16, 12)
第20天
新的最大收益: 8275.0
20天
新的最大收益状态: (27, 8275, 0, 0)
新的最大收益前一个状态: (21, 8275, 16, 12)
第21天
新的最大收益: 8280.0
21天
新的最大收益状态: (27, 8110, 68, 0)
新的最大收益前一个状态: (21, 8110, 78, 14)
新的最大收益: 8292.5
21天
新的最大收益状态: (27, 8125, 67, 0)
新的最大收益前一个状态: (21, 8125, 77, 14)
新的最大收益: 8315.0
21天
新的最大收益状态: (27, 8150, 66, 0)
新的最大收益前一个状态: (21, 8150, 76, 14)
新的最大收益: 8327.5
21天
新的最大收益状态: (27, 8165, 65, 0)
新的最大收益前一个状态: (21, 8165, 75, 14)
新的最大收益: 8350.0
21天
新的最大收益状态: (27, 8190, 64, 0)
新的最大收益前一个状态: (21, 8190, 74, 14)
新的最大收益: 8362.5
21天
新的最大收益状态: (27, 8205, 63, 0)
新的最大收益前一个状态: (21, 8205, 73, 14)
新的最大收益: 8385.0
21天
新的最大收益状态: (27, 8230, 62, 0)
新的最大收益前一个状态: (21, 8230, 72, 14)
新的最大收益: 8397.5
21天
新的最大收益状态: (27,

In [ ]:
for key, value in dp[24].items():
    if key[0] == 27 and key[1] == 0 and key[2] == 0:
        print(key, value)



(27, 0, 0) (10470, 21, 10470, 10, 14)


In [38]:
# 从您图片中显示的最终状态开始
day = 24 
# 这是您要查询的初始 key
current_key = (27, 0, 0) 

# 检查 dp 表和初始 key 是否存在
if not 'dp' in locals() or not isinstance(dp, list) or len(dp) <= day:
    print("错误: 'dp' 列表不存在或不完整。请先运行您的模拟代码来填充 'dp' 表。")
elif current_key not in dp[day]:
    print(f"错误: 在 dp[{day}] 中找不到初始状态 {current_key}。")
else:
    # 获取最终状态的 value
    current_value = dp[day][current_key]
    
    # 用于存储路径
    path = [(current_key, current_value)]

    # 从第 23 天开始向前回溯
    for d in range(day - 1, -1, -1):
        # 根据您描述的逻辑，从当前 value 构建前一天的 key
        # value[1] 是前一天的 pos
        # value[3] 是前一天的 water
        # value[4] 是前一天的 food
        # 注意：这里的索引基于您图片中 value 的结构 (money, pre_pos, pre_money, pre_water, pre_food)
        prev_key = (current_value[1], current_value[3], current_value[4])
        
        # 在前一天的 dp 表中查找
        if prev_key in dp[d]:
            # 找到了，更新当前状态并记录路径
            current_key = prev_key
            current_value = dp[d][current_key]
            path.append((current_key, current_value))
        else:
            # 如果找不到，说明路径中断
            print(f"路径在第 {d} 天中断，无法找到状态: {prev_key}")
            break
            
    # 打印完整的回溯路径（从开始到结束）
    print("--- 最优路径回溯结果 ---")
    # 我们从后往前找的，所以需要反转列表
    for i, (key, value) in enumerate(reversed(path)):
        # 这里的天数是模拟的天数（从1开始）
        day_num = day - len(path) + i + 2
        print(f"第 {day_num-1} 天: 状态={key}, 值={value}")

--- 最优路径回溯结果 ---
第 0 天: 状态=(1, 178, 333), 值=(5780, None, None, None, None)
第 1 天: 状态=(25, 162, 321), 值=(5780, 1, 5780, 178, 333)
第 2 天: 状态=(26, 146, 309), 值=(5780, 25, 5780, 162, 321)
第 3 天: 状态=(23, 136, 295), 值=(5780, 26, 5780, 146, 309)
第 4 天: 状态=(23, 126, 285), 值=(5780, 23, 5780, 136, 295)
第 5 天: 状态=(21, 116, 271), 值=(5780, 23, 5780, 126, 285)
第 6 天: 状态=(9, 100, 259), 值=(5780, 21, 5780, 116, 271)
第 7 天: 状态=(9, 90, 249), 值=(5780, 9, 5780, 100, 259)
第 8 天: 状态=(15, 243, 235), 值=(4150, 9, 5780, 90, 249)
第 9 天: 状态=(13, 227, 223), 值=(4150, 15, 4150, 243, 235)
第 10 天: 状态=(12, 211, 211), 值=(4150, 13, 4150, 227, 223)
第 11 天: 状态=(12, 201, 201), 值=(4150, 12, 4150, 211, 211)
第 12 天: 状态=(12, 177, 183), 值=(5150, 12, 4150, 201, 201)
第 13 天: 状态=(12, 162, 162), 值=(6150, 12, 5150, 177, 183)
第 14 天: 状态=(12, 138, 144), 值=(7150, 12, 6150, 162, 162)
第 15 天: 状态=(12, 114, 126), 值=(8150, 12, 7150, 138, 144)
第 16 天: 状态=(12, 90, 108), 值=(9150, 12, 8150, 114, 126)
第 17 天: 状态=(12, 80, 98), 值=(9150, 12, 9150, 90